# Phase 4 — A3 | Knowledge Distillation (Student)

Here I use the trained A3 Phase 4 teacher to guide a smaller A3 student. The teacher notebook has to finish first.

I am keeping each method in its own Kaggle notebook so that its training history, checkpoint, predictions and Hugging Face backup stay separate. The main flow is: load the Phase 1 model → check the Phase 4 data → train → evaluate → predict → prepare the submission → back up the results.

**Status:** These cells are ready to run, but real training scores and cloud uploads only exist after I execute them in Kaggle.


## 1. Packages

I keep package installation separate from the rest of the experiment, so it is easy to rerun if Kaggle starts with a clean environment.

In [1]:
%pip -q install "transformers>=4.45,<5" "huggingface_hub>=0.31,<2" "safetensors>=0.4.5" scikit-learn tqdm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 101.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 85.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


## 2. Run settings

The method is fixed in this notebook. I only change compute or experiment settings here; different adaptation methods have their own notebooks.

In [2]:
# This file is for one experiment, so I keep its method fixed.
ARCH = 'a3'
METHOD = 'kd'    # One method per notebook

# My Phase 1 checkpoint locations from the earlier notebooks.
# The Hugging Face username is read from the token when I log in.
SOURCE_REPO_NAME = 'cs781-phase1-architecture3'
CHECKPOINT_FILE = 'model_state.pt'
# The original checkpoint may belong to a different Hugging Face account:
SOURCE_REPO_OVERRIDE = 'P-261110044/cs781-phase1-architecture3'  # Source owner from the Phase 2 notebook
TEACHER_REPO_NAME = 'cs781-phase4-a3-full'  # Only the A3 student uses this teacher repo

CHUNK_LENGTH = 192          # <= base model's positional/context limit
MAX_CHUNKS = 3             # evenly spaced: covers start, middle and end
EPOCHS, PATIENCE = 3, 2  # Stop early if dev Macro-F1 does not improve
GRAD_ACCUM = 4
LR_BACKBONE, LR_HEAD = 2e-5, 2e-4
LR_ADAPT = 1e-4
LORMA_RANK, LORMA_ALPHA = 8, 16
ADAPTER_WIDTH = 48
STUDENT_LAYERS = 4          # only used for A3/KD
KD_T = 2.0
SEED = 42

# Leaving these as None keeps all official examples (needed for submission).
MAX_TRAIN = MAX_DEV = MAX_TEST = None
UPLOAD_TO_HF = True        # Separate upload cell; runs after files are created
LOCAL_DATA_ROOT = ''       # Folder containing Phase_4_CJPE, if attached on Kaggle
LOCAL_MODEL_ROOT = ''      # Local copy of the Phase 1 repo, if available
# Macro-F1 is the main metric and treats both classes equally.
# Class weights come from the training labels only.
USE_CLASS_WEIGHTS = True

## 3. Imports and output folder

I set the random seed, select the available GPU and create a folder for this experiment. Its files do not share a folder with the other runs.

In [3]:
# Keep the seed, device and output folder together for this run.
import os, json, re, math, time, zipfile, random, shutil, gc
from pathlib import Path
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix
from transformers import PreTrainedTokenizerFast
from huggingface_hub import HfApi, hf_hub_download
from safetensors.torch import load_file
from tqdm.auto import tqdm

assert METHOD in ('full', 'kd'), f'Unsupported method for architecture {ARCH}: {METHOD}'
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT = Path('/kaggle/working' if Path('/kaggle/working').is_dir() else '.') / f'phase4_{ARCH}_{METHOD}'
OUT.mkdir(exist_ok=True,parents=True)

# Kaggle secret name may be HF_TOKEN or HF; neither the token nor its value is printed.


## 4. Hugging Face login

I read `HF_TOKEN` from Kaggle Secrets and get the account name from Hugging Face. No username or access token needs to be pasted into the notebook.

In [4]:
# Read the token from Kaggle Secrets. The source repo is my Phase 1 checkpoint.
TOKEN = os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')
if not TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        for key in ('HF_TOKEN','HF'):
            try:
                TOKEN = secrets.get_secret(key)
                if TOKEN: break
            except Exception: pass
    except ImportError: pass
if not TOKEN:
    raise RuntimeError('Set Kaggle Secret HF_TOKEN (with HF read/write permissions). Internet must be on.')
api = HfApi(token=TOKEN)
HF_USER = api.whoami()['name']        # automatic identity, never hardcoded
P1_REPO = SOURCE_REPO_OVERRIDE or f'{HF_USER}/{SOURCE_REPO_NAME}'
OUT_REPO = f'{HF_USER}/cs781-phase4-{ARCH}-{METHOD}'
print('Source:', P1_REPO, 'Output repo:', OUT_REPO, 'Device:', DEVICE)


Source: P-261110044/cs781-phase1-architecture3 Output repo: P-261110044/cs781-phase4-a3-kd Device: cuda


## 5. Phase 1 model architecture

This is the original A3 architecture used for Phase 1. I keep its layers and checkpoint parameter names unchanged so the saved weights can be restored.

In [5]:
# Copied from a3_distillation.ipynb; no pretrained external model substituted.
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        # Compute the RMS in fp32 for numerical stability, then cast back.
        rms = x.float().pow(2).mean(dim=-1, keepdim=True)
        x_norm = x * torch.rsqrt(rms + self.eps).to(x.dtype)
        return x_norm * self.weight

def rotate_half(x):
    # RoPE rotates each consecutive pair: (x0, x1) -> (-x1, x0).
    x_even = x[..., ::2]
    x_odd = x[..., 1::2]
    return torch.stack((-x_odd, x_even), dim=-1).flatten(-2)

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, base=10000.0):
        super().__init__()
        assert head_dim % 2 == 0, "RoPE needs an even head dimension."
        inv_freq = 1.0 / (
            base ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim)
        )
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def cos_sin(self, positions, dtype):
        # positions: [seq]
        freqs = torch.outer(positions.float(), self.inv_freq)
        cos = freqs.cos().repeat_interleave(2, dim=-1).to(dtype)
        sin = freqs.sin().repeat_interleave(2, dim=-1).to(dtype)
        return cos[None, None, :, :], sin[None, None, :, :]

    def apply_rotary(self, q, k, positions):
        cos, sin = self.cos_sin(positions, q.dtype)
        q = (q * cos) + (rotate_half(q) * sin)
        k = (k * cos) + (rotate_half(k) * sin)
        return q, k

class GroupedQueryLocalAttention(nn.Module):
    def __init__(
        self,
        dim,
        num_heads,
        num_kv_heads,
        window_size,
        chunk_size=256,
        dropout=0.1,
    ):
        super().__init__()
        assert dim % num_heads == 0
        assert num_heads % num_kv_heads == 0

        self.dim = dim
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = dim // num_heads
        self.groups = num_heads // num_kv_heads
        self.window_size = window_size
        self.chunk_size = chunk_size
        self.dropout = dropout

        self.q_proj = nn.Linear(dim, num_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(dim, num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(dim, num_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(num_heads * self.head_dim, dim, bias=False)

        self.rope = RotaryEmbedding(self.head_dim)

    def _repeat_kv(self, x):
        # [B, kv_heads, T, D] -> [B, query_heads, T, D]
        return x.repeat_interleave(self.groups, dim=1)

    def _local_attention(self, q, k, v):
        """
        Exact causal sliding-window attention.

        We process a few query positions at a time.  Each query chunk only
        materializes the K/V region that can actually be attended to, instead
        of creating a full 2048 x 2048 attention matrix.
        """
        B, H, T, D = q.shape
        outputs = []

        k_rep = self._repeat_kv(k)
        v_rep = self._repeat_kv(v)

        for q_start in range(0, T, self.chunk_size):
            q_end = min(q_start + self.chunk_size, T)

            # Earliest key any query in this chunk could legally use.
            k_start = max(0, q_start - self.window_size + 1)
            k_end = q_end

            q_chunk = q[:, :, q_start:q_end, :]
            k_chunk = k_rep[:, :, k_start:k_end, :]
            v_chunk = v_rep[:, :, k_start:k_end, :]

            q_pos = torch.arange(q_start, q_end, device=q.device)
            k_pos = torch.arange(k_start, k_end, device=q.device)

            # True = allowed attention position.
            causal = k_pos[None, :] <= q_pos[:, None]
            within_window = k_pos[None, :] >= (
                q_pos[:, None] - self.window_size + 1
            )
            attn_mask = (causal & within_window)[None, None, :, :]

            out = F.scaled_dot_product_attention(
                q_chunk,
                k_chunk,
                v_chunk,
                attn_mask=attn_mask,
                dropout_p=self.dropout if self.training else 0.0,
                is_causal=False,
            )
            outputs.append(out)

        return torch.cat(outputs, dim=2)

    def forward(
        self,
        x,
        past_kv=None,
        use_cache=False,
        position_offset=0,
    ):
        B, T, _ = x.shape

        q = self.q_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)

        positions = torch.arange(
            position_offset,
            position_offset + T,
            device=x.device,
        )
        q, k = self.rope.apply_rotary(q, k, positions)

        if past_kv is None:
            # Training/prefill path: causal local attention over this sequence.
            attn_out = self._local_attention(q, k, v)
            k_all, v_all = k, v
        else:
            # Generation path.  We decode one token at a time after prefill.
            assert T == 1, "Cached decoding expects one new token at a time."
            past_k, past_v = past_kv
            k_all = torch.cat([past_k, k], dim=2)
            v_all = torch.cat([past_v, v], dim=2)

            # Keep only the most recent local window in the KV cache.
            k_all = k_all[:, :, -self.window_size:, :]
            v_all = v_all[:, :, -self.window_size:, :]

            k_rep = self._repeat_kv(k_all)
            v_rep = self._repeat_kv(v_all)

            # Every cached token is in the past (or the current token), so no
            # extra causal mask is necessary for this one-token decode step.
            attn_out = F.scaled_dot_product_attention(
                q,
                k_rep,
                v_rep,
                dropout_p=0.0,
                is_causal=False,
            )

        new_kv = None
        if use_cache:
            new_kv = (
                k_all[:, :, -self.window_size:, :],
                v_all[:, :, -self.window_size:, :],
            )

        attn_out = attn_out.transpose(1, 2).contiguous().view(B, T, self.dim)
        return self.o_proj(attn_out), new_kv

class SwiGLU(nn.Module):
    def __init__(self, dim, hidden_dim, dropout=0.1):
        super().__init__()
        self.gate_proj = nn.Linear(dim, hidden_dim, bias=False)
        self.up_proj = nn.Linear(dim, hidden_dim, bias=False)
        self.down_proj = nn.Linear(hidden_dim, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = F.silu(self.gate_proj(x)) * self.up_proj(x)
        return self.dropout(self.down_proj(x))

class ModernTransformerBlock(nn.Module):
    def __init__(
        self,
        dim,
        num_heads,
        num_kv_heads,
        ffn_dim,
        window_size,
        chunk_size,
        dropout,
    ):
        super().__init__()
        self.attn_norm = RMSNorm(dim)
        self.attn = GroupedQueryLocalAttention(
            dim=dim,
            num_heads=num_heads,
            num_kv_heads=num_kv_heads,
            window_size=window_size,
            chunk_size=chunk_size,
            dropout=dropout,
        )
        self.ffn_norm = RMSNorm(dim)
        self.ffn = SwiGLU(dim, ffn_dim, dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x, past_kv=None, use_cache=False, position_offset=0):
        attn_out, new_kv = self.attn(
            self.attn_norm(x),
            past_kv=past_kv,
            use_cache=use_cache,
            position_offset=position_offset,
        )
        x = x + self.resid_dropout(attn_out)
        x = x + self.ffn(self.ffn_norm(x))
        return x, new_kv

class ModernLocalTransformerLM(nn.Module):
    def __init__(
        self,
        vocab_size,
        dim=600,
        num_layers=8,
        num_heads=10,
        num_kv_heads=5,
        ffn_dim=1600,
        window_size=512,
        chunk_size=256,
        dropout=0.1,
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.dim = dim
        self.num_layers = num_layers

        self.token_embedding = nn.Embedding(vocab_size, dim)
        self.embed_dropout = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            ModernTransformerBlock(
                dim=dim,
                num_heads=num_heads,
                num_kv_heads=num_kv_heads,
                ffn_dim=ffn_dim,
                window_size=window_size,
                chunk_size=chunk_size,
                dropout=dropout,
            )
            for _ in range(num_layers)
        ])

        self.final_norm = RMSNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size, bias=False)

        # Weight tying saves parameters and is standard for causal LMs.
        self.lm_head.weight = self.token_embedding.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def _run_hidden(
        self,
        input_ids,
        past_key_values=None,
        use_cache=False,
        position_offset=0,
    ):
        x = self.embed_dropout(self.token_embedding(input_ids))
        new_caches = [] if use_cache else None

        for i, block in enumerate(self.blocks):
            past = None if past_key_values is None else past_key_values[i]
            x, new_kv = block(
                x,
                past_kv=past,
                use_cache=use_cache,
                position_offset=position_offset,
            )
            if use_cache:
                new_caches.append(new_kv)

        x = self.final_norm(x)
        return x, new_caches

    def forward(self, input_ids, labels=None, attention_mask=None, **kwargs):
        x, _ = self._run_hidden(input_ids)
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            # Position t predicts token t+1.
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
            )

        return CausalLMOutput(loss=loss, logits=logits)

    @torch.inference_mode()
    def prefill(self, input_ids):
        x, caches = self._run_hidden(
            input_ids,
            past_key_values=None,
            use_cache=True,
            position_offset=0,
        )
        return self.lm_head(x), caches

    @torch.inference_mode()
    def decode_step(self, input_ids, past_key_values, position_offset):
        x, caches = self._run_hidden(
            input_ids,
            past_key_values=past_key_values,
            use_cache=True,
            position_offset=position_offset,
        )
        return self.lm_head(x[:, -1, :]), caches


## 6. Phase 1 checkpoint and tokenizer

I find the Phase 1 checkpoint and its `tokenizer.json` in the source repository. The tokenizer uses the same `<pad>`, `<unk>`, `<s>` and `</s>` tokens as my Phase 1 training notebook. I also compare its vocabulary size with the checkpoint embeddings.

In [6]:
# Find source files, copy the original tokenizer, and read the exact saved weights.
# ---- Official Phase-1 checkpoint; exact architecture and vocabulary ----
repo_files = ([str(f.relative_to(LOCAL_MODEL_ROOT)).replace('\\','/')
               for f in Path(LOCAL_MODEL_ROOT).rglob('*') if f.is_file()]
              if LOCAL_MODEL_ROOT else api.list_repo_files(P1_REPO, repo_type='model'))
if CHECKPOINT_FILE not in repo_files:
    raise FileNotFoundError(f'Expected {CHECKPOINT_FILE} in {P1_REPO}, available: {repo_files[:40]}')

def fetch(rel):
    return (Path(LOCAL_MODEL_ROOT)/rel if LOCAL_MODEL_ROOT else
            Path(hf_hub_download(repo_id=P1_REPO,filename=rel,token=TOKEN)))

parent = str(Path(CHECKPOINT_FILE).parent)
parent = '' if parent == '.' else parent+'/'
folders = list(dict.fromkeys([parent,'','final_model/','architecture-3-final/',
                              'architecture-4-final/', 'cs781_phase1_a2_encoderdecoder/legal-gpt2-final/']))
tokenizer_folder = next((f for f in folders if f+'tokenizer.json' in repo_files),None)
if tokenizer_folder is None: raise FileNotFoundError('Original Phase-1 tokenizer.json not found')
tok_dir = OUT/'tokenizer'; tok_dir.mkdir(exist_ok=True)
for name in ('tokenizer.json','tokenizer_config.json','special_tokens_map.json','added_tokens.json'):
    if tokenizer_folder+name in repo_files:
        shutil.copy2(fetch(tokenizer_folder+name),tok_dir/name)
# I load the exact tokenizer JSON used in Phase 1.  Loading this file directly
# keeps the original special-token setup and avoids the tokenizer-class warning.
tokenizer = PreTrainedTokenizerFast(
    tokenizer_file=str(tok_dir / "tokenizer.json"),
    pad_token="<pad>",
    unk_token="<unk>",
    bos_token="<s>",
    eos_token="</s>",
)
print("Tokenizer ready:", len(tokenizer), "tokens")
print("Special token IDs:", {
    "pad": tokenizer.pad_token_id,
    "unk": tokenizer.unk_token_id,
    "bos": tokenizer.bos_token_id,
    "eos": tokenizer.eos_token_id,
})
# I do not add new vocabulary entries because the checkpoint size must stay unchanged.
if tokenizer.pad_token_id is None:
    raise ValueError("The Phase 1 tokenizer has no pad token")

config_name = next((n for n in (parent+'architecture_config.json','architecture_config.json') if n in repo_files),None)
saved_cfg = json.loads(fetch(config_name).read_text()) if config_name else {}
pth = fetch(CHECKPOINT_FILE)
state = load_file(str(pth),device='cpu') if pth.suffix=='.safetensors' else torch.load(pth,map_location='cpu',weights_only=True)
for key in ('model_state','state_dict','model_state_dict'):
    if key in state and isinstance(state[key],dict): state=state[key]; break
for prefix in ('module.','_orig_mod.'):
    if state and all(k.startswith(prefix) for k in state):
        state={k[len(prefix):]:v for k,v in state.items()}
if 'token_embedding.weight' not in state:
    raise ValueError('Checkpoint does not have the Phase-1 token_embedding.weight key')
VOCAB,DIM = state['token_embedding.weight'].shape
if len(tokenizer) != VOCAB: raise ValueError(f'Tokenizer mismatch {len(tokenizer)} vs checkpoint {VOCAB}')


tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

Tokenizer ready: 50000 tokens
Special token IDs: {'pad': 0, 'unk': 1, 'bos': 2, 'eos': 3}


architecture_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

model_state.pt:   0%|          | 0.00/247M [00:00<?, ?B/s]

## 7. Restore the pretrained backbone

I rebuild the backbone using the saved dimensions and load every expected weight. A mismatch should stop here rather than produce a model with partly random parameters.

In [7]:
# Rebuild the original backbone and check every checkpoint weight.
def layer_count(prefix):
    found={int(m.group(1)) for k in state if (m:=re.match(re.escape(prefix)+r'\.(\d+)\.',k))}
    if not found or found != set(range(max(found)+1)): raise ValueError(f'Invalid {prefix} blocks: {found}')
    return len(found)

def heads(arch,dim,n):
    recorded = saved_cfg.get('num_heads',saved_cfg.get('n_heads'))
    if recorded: return int(recorded)
    if arch=='a1' and (dim,n)==(640,8): return 10
    if arch=='a2' and (dim,n)==(600,8): return 15
    raise ValueError('Attention head count missing from config; add num_heads to Phase-1 architecture_config.json')

def make_backbone(student_layers=None):
    needed=('num_heads','num_kv_heads','window_size','chunk_size')
    if any(k not in saved_cfg for k in needed):
        raise ValueError('Architecture-3 config missing: '+str([k for k in needed if k not in saved_cfg]))
    n = layer_count('blocks') if student_layers is None else student_layers
    return ModernLocalTransformerLM(vocab_size=VOCAB,dim=DIM,num_layers=n,
        num_heads=int(saved_cfg['num_heads']),num_kv_heads=int(saved_cfg['num_kv_heads']),
        ffn_dim=state['blocks.0.ffn.gate_proj.weight'].shape[0],
        window_size=int(saved_cfg['window_size']),chunk_size=int(saved_cfg['chunk_size']),
        dropout=float(saved_cfg.get('dropout',0.1)))

def strict_load(m, student_layers=None):
    target = m.state_dict()
    sub = {k:v for k,v in state.items() if k in target} if student_layers else dict(state)
    for a,b in (('lm_head.weight','token_embedding.weight'),('token_embedding.weight','lm_head.weight')):
        if a in target and a not in sub and b in sub and target[a].shape == sub[b].shape:
            sub[a] = sub[b]
    if 'lm_head.weight' in sub and 'lm_head.weight' not in target and 'output_layer.weight' in target:
        sub.setdefault('output_layer.weight', sub.pop('lm_head.weight'))
    missing = set(target)-set(sub); extra = set(sub)-set(target)
    invalid = [k for k in set(target)&set(sub) if target[k].shape != sub[k].shape]
    if missing or extra or invalid:
        raise ValueError(f'Strict checkpoint mismatch: missing={sorted(missing)[:5]}, extra={sorted(extra)[:5]}, shape={invalid[:5]}')
    m.load_state_dict(sub,strict=True)
    if ARCH in ('a1','a2'): m.output_layer=nn.Identity()
    else: m.lm_head=nn.Identity()
    return m

backbone = strict_load(make_backbone())
print('Exact Phase-1 backbone loaded:',ARCH,'width:',DIM,'vocab:',VOCAB)


Exact Phase-1 backbone loaded: a3 width: 600 vocab: 50000


## 8. Phase 4 dataset and duplicate checks

I use the official Phase 4 train, dev and test files. The train split can contain repeated case IDs (for example `2019_890`), so I retain those rows and report them instead of failing. I also check whether IDs or identical text appear across the different splits. Those overlaps need investigation before trusting a score.

In [8]:
# Official Phase 4 files. Repeated TRAIN case IDs can occur when several
# instances/passages of the same case are included in the training split.
# We preserve these records for inspection and training; repeated IDs by
# themselves are NOT proof of train/test leakage.
import hashlib
from collections import defaultdict

DATA_REPO = 'Exploration-Lab/CS-781-Capstone'
paths = None

# Pin all 3 splits to ONE revision, so the files cannot change mid-download.
# When using attached local files, the revision is not needed.
DATA_REVISION = api.dataset_info(DATA_REPO).sha if not LOCAL_DATA_ROOT else None
print('Official dataset revision:', DATA_REVISION or '(local Kaggle dataset)')

def data_file(split):
    name = f'{split}.json'
    folder = 'Test' if split == 'test' else 'Train'
    if LOCAL_DATA_ROOT:
        root = Path(LOCAL_DATA_ROOT)
        options = (root/'Phase_4_CJPE'/folder/name, root/folder/name, root/name)
        for p in options:
            if p.is_file(): return p
        raise FileNotFoundError(f'Official Phase 4 file not found: {options}')

    global paths
    if paths is None:
        paths = api.list_repo_files(DATA_REPO, repo_type='dataset', revision=DATA_REVISION)
    matches = [p for p in paths if ('phase_4' in p.lower()
               and p.lower().endswith('/'+folder.lower()+'/'+name))]
    if len(matches) != 1:
        raise FileNotFoundError(f'Expected one {folder}/{name}; found {matches}')
    return Path(hf_hub_download(repo_id=DATA_REPO,filename=matches[0],
              repo_type='dataset',revision=DATA_REVISION,token=TOKEN))

def load_cases(split):
    """Preserve every training instance; require test IDs to be unique."""
    raw = json.loads(data_file(split).read_text(encoding='utf-8'))
    if isinstance(raw, dict) and isinstance(raw.get('data'), list): raw = raw['data']
    if isinstance(raw, dict):
        raw = [dict(v,id=k) if isinstance(v,dict) else {'id':k,'text':v}
               for k,v in raw.items()]
    if not isinstance(raw, list): raise ValueError(f'{split}: expected a list or ID-keyed dictionary')
    out,seen=[],set()
    for item in raw:
        if 'id' not in item or 'text' not in item:
            raise ValueError(f'{split}: missing id/text; keys={list(item)}')
        ident = str(item['id']).strip()
        if not ident or ident.lower()=='none':
            raise ValueError(f'{split}: empty/invalid case ID')
        if split == 'test' and ident in seen:
            raise ValueError(f'test: ID {ident} repeats; cannot submit one prediction per record')
        seen.add(ident)
        txt=item['text']
        if isinstance(txt,list): txt='\n'.join(txt)
        if not isinstance(txt,str) or not txt.strip():
            raise ValueError(f'{split}: invalid text for {ident}')
        case={'id':ident,'text':txt}
        if split != 'test':
            keys=[k for k in ('label','labels','outcome','judgment','judgement','target','y') if k in item]
            if len(keys)!=1:
                raise ValueError(f'{split} {ident}: expected one binary label key; found {keys}; keys={list(item)}')
            y=item[keys[0]]
            if type(y)!=int or y not in (0,1):
                raise ValueError(f'{split} {ident}: expected integer 0/1; got {y!r}')
            case['label']=y
        out.append(case)
    return out

# First load all data, THEN audit repeated IDs and overlap across splits.
train_cases,dev_cases,test_cases = (load_cases(s) for s in ('train','dev','test'))

# Within-split duplicates are reported but do not stop TRAIN from loading.
for split, cases in [('train',train_cases),('dev',dev_cases),('test',test_cases)]:
    counts=Counter(c['id'] for c in cases)
    duplicates={k:v for k,v in counts.items() if v>1}
    print(f'{split}: {len(cases)} rows, {len(counts)} unique IDs, '
          f'{len(duplicates)} repeated IDs, {sum(v-1 for v in duplicates.values())} extra rows')
    if duplicates:
        print('  Examples:',list(duplicates.items())[:5])
        if split=='dev':
            print('  WARNING: Dev contains multiple records for a case; check metric interpretation.')
    if split!='test':
        labels=defaultdict(set)
        for c in cases: labels[c['id']].add(c['label'])
        conflicting=[k for k,v in labels.items() if len(v)>1]
        if conflicting:
            print(f'  WARNING: {len(conflicting)} case IDs have conflicting labels, e.g. {conflicting[:5]}')

ids={name:set(c['id'] for c in cases) for name,cases in
     [('train',train_cases),('dev',dev_cases),('test',test_cases)]}
for a,b in [('train','dev'),('train','test'),('dev','test')]:
    shared=ids[a]&ids[b]
    print(f'{a}/{b} shared case IDs: {len(shared)}',sorted(shared)[:5])
    if shared:
        raise ValueError(f'POTENTIAL LEAKAGE: {len(shared)} IDs shared by {a} and {b}. '
                         'Inspect the standalone dataset-audit notebook before training.')

# Exact same text under DIFFERENT IDs is also potential leakage.
# Report hashes (rather than comparing hundreds of thousands of text strings).
def text_hash(x):
    normalized=' '.join(x.lower().split())
    return hashlib.sha256(normalized.encode('utf-8')).hexdigest()
texts={name:set(text_hash(c['text']) for c in cases) for name,cases in
       [('train',train_cases),('dev',dev_cases),('test',test_cases)]}
for a,b in [('train','dev'),('train','test'),('dev','test')]:
    shared=texts[a]&texts[b]
    print(f'{a}/{b} identical normalized texts: {len(shared)}')
    if shared:
        raise ValueError(f'POTENTIAL LEAKAGE: {len(shared)} matching texts between {a}/{b}. '
                         'Inspect the standalone dataset-audit notebook before training.')

TEST_IDS=[c['id'] for c in test_cases]
print('Training label counts:',Counter(c['label'] for c in train_cases))
# Only cap AFTER auditing all official records. Never submit capped test predictions.
train_cases=train_cases[:MAX_TRAIN] if MAX_TRAIN is not None else train_cases
dev_cases=dev_cases[:MAX_DEV] if MAX_DEV is not None else dev_cases
test_cases=test_cases[:MAX_TEST] if MAX_TEST is not None else test_cases


Official dataset revision: 93b0059be756c6c461f3a225de25aabda7d5f84a


Phase_4_CJPE/Train/train.json:   0%|          | 0.00/737M [00:00<?, ?B/s]

Phase_4_CJPE/Train/dev.json:   0%|          | 0.00/45.0M [00:00<?, ?B/s]

Phase_4_CJPE/Test/test.json:   0%|          | 0.00/35.3M [00:00<?, ?B/s]

train: 37387 rows, 32305 unique IDs, 5082 repeated IDs, 5082 extra rows
  Examples: [('2019_890', 2), ('2014_170', 2), ('2010_721', 2), ('2008_1460', 2), ('2008_188', 2)]
dev: 1988 rows, 994 unique IDs, 994 repeated IDs, 994 extra rows
  Examples: [('1989_75', 2), ('1959_102', 2), ('1979_76', 2), ('1991_124', 2), ('1985_233', 2)]
test: 1517 rows, 1517 unique IDs, 0 repeated IDs, 0 extra rows
train/dev shared case IDs: 0 []
train/test shared case IDs: 0 []
dev/test shared case IDs: 0 []
train/dev identical normalized texts: 0
train/test identical normalized texts: 0
dev/test identical normalized texts: 0
Training label counts: Counter({0: 22067, 1: 15320})


## 9. Preparing chunks from long cases

Some judgments are longer than the model context. I take a few chunks spread across the case, keeping the beginning, middle and end when possible. The predicted label still belongs to the entire case.

In [9]:
# Tokenize each case once and keep a few chunks across the document.
# Pre-tokenize exactly once per case, keep first and last document information.
if ARCH=='a1': model_context=backbone.context_length
elif ARCH=='a2': model_context=backbone.source_position.num_embeddings
elif ARCH=='a4': model_context=backbone.context_length
else: model_context=CHUNK_LENGTH  # A3 implements internal local windows
if CHUNK_LENGTH > model_context: raise ValueError('CHUNK_LENGTH exceeds source model context')

def chunks(text):
    ids=tokenizer.encode(text,add_special_tokens=False)
    if not ids: ids=[tokenizer.unk_token_id]
    windows=[ids[i:i+CHUNK_LENGTH] for i in range(0,len(ids),CHUNK_LENGTH)]
    if len(windows)>MAX_CHUNKS:
        idx=np.linspace(0,len(windows)-1,MAX_CHUNKS,dtype=int).tolist()
        windows=[windows[i] for i in idx]
    return [torch.tensor(x,dtype=torch.long) for x in windows]

# Preserve official case order for submission and dev metrics.
train_data=[(chunks(x['text']),x.get('label'),x['id']) for x in train_cases]
dev_data=[(chunks(x['text']),x.get('label'),x['id']) for x in dev_cases]
test_data=[(chunks(x['text']),None,x['id']) for x in test_cases]


## 10. Case classifier

I average representations from the selected chunks and use a small two-class head. The head predicts the dataset labels `0` and `1`; it does not assume either label has a particular legal meaning.

In [10]:
# Add the classifier and keep only the selected parameters trainable.
# ---- Classification head + source-inspired adaptation methods ----
class Adapter(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.down=nn.Linear(width,ADAPTER_WIDTH)
        self.up=nn.Linear(ADAPTER_WIDTH,width)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self,x): return self.up(F.gelu(self.down(x)))

class LoRMALinear(nn.Module):
    # Matches the Phase-2 A4 method: multiplicative low-rank correction on W(x).
    def __init__(self,original):
        super().__init__()
        self.original=original
        self.original.requires_grad_(False)
        self.A=nn.Linear(original.out_features,LORMA_RANK,bias=False)
        self.B=nn.Linear(LORMA_RANK,original.out_features,bias=False)
        nn.init.zeros_(self.B.weight)
    def forward(self,x):
        z=self.original(x)
        return z+self.B(self.A(z))*(LORMA_ALPHA/LORMA_RANK)

class Classifier(nn.Module):
    def __init__(self,base,method):
        super().__init__()
        self.base=base; self.method=method
        self.head=nn.Linear(DIM,2)
        self._hooks=[]
        if method not in ('full','kd'): self.base.requires_grad_(False)
        blocks=self.base.encoder_blocks if ARCH=='a2' else self.base.blocks
        if method=='partial':
            for layer in blocks[-2:]: layer.requires_grad_(True)
        elif method=='bitfit':
            for n,p in self.base.named_parameters():
                if n.endswith('bias'): p.requires_grad_(True)
        elif method=='adapters':
            self.adapters=nn.ModuleList([Adapter(DIM) for _ in blocks])
            for block,adapter in zip(blocks,self.adapters):
                def add_adapter(mod,args,out,adapter=adapter):
                    if isinstance(out,tuple): return (out[0]+adapter(out[0]),*out[1:])
                    return out+adapter(out)
                self._hooks.append(block.register_forward_hook(add_adapter))
        elif method=='lorma':
            for block in blocks:
                block.attn.q_proj=LoRMALinear(block.attn.q_proj)
                block.attn.v_proj=LoRMALinear(block.attn.v_proj)

    def feature(self,ids):
        # One chunk -> one contextual representation. Only the correct backbone method is called.
        x=ids.to(DEVICE).unsqueeze(0)
        if ARCH=='a1': hidden=self.base(x).logits
        elif ARCH=='a2': hidden=self.base.encode(x)
        else: hidden,_=self.base._run_hidden(x)
        return hidden[:,-1,:].squeeze(0) if ARCH!='a2' else hidden.mean(1).squeeze(0)

    def forward(self,parts):
        # Average chunk vectors so the final decision is at CASE level, not sentence level.
        features=torch.stack([self.feature(x) for x in parts])
        return self.head(features.mean(0))


## 11. Set up this experiment

I first load the already fine-tuned A3 teacher, check its settings, then create a smaller student. The teacher provides soft target logits only for training examples.

In [11]:
# Use the trained Phase 4 A3 teacher, not a fresh classifier.
# The A3 teacher notebook has to finish and upload before this run.
teacher_repo = f'{HF_USER}/{TEACHER_REPO_NAME}'
file = Path(hf_hub_download(repo_id=teacher_repo, filename='best_model.pt', token=TOKEN))
teacher_ckpt = torch.load(file, map_location='cpu', weights_only=True)
if teacher_ckpt['arch'] != 'a3' or teacher_ckpt['method'] != 'full':
    raise ValueError('The teacher must be a Phase-4 A3 full fine-tuned classifier.')
if (teacher_ckpt['chunk_length'], teacher_ckpt['max_chunks']) != (CHUNK_LENGTH, MAX_CHUNKS):
    raise ValueError('The teacher and student must have identical text-chunk settings.')

# Build the teacher with the pretrained Phase-1 A3 backbone.
teacher = Classifier(backbone, 'full').to(DEVICE)
teacher.load_state_dict(teacher_ckpt['model'], strict=True)
teacher.eval()

# Precompute teacher's classification logits for training cases ONLY.
# No development or test labels are used in knowledge distillation.
with torch.no_grad():
    soft_targets = [teacher(parts).float().cpu() for parts, _, _ in
                    tqdm(train_data, desc='Teacher logits')]

# Free GPU memory before building the smaller student.
del teacher, backbone, teacher_ckpt
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
if STUDENT_LAYERS >= layer_count('blocks'):
    raise ValueError('Student must contain fewer Transformer blocks than the teacher.')

# Copy compatible first layers from the user's own Phase-1 pretrained weights.
student_backbone = strict_load(make_backbone(STUDENT_LAYERS),
                               student_layers=STUDENT_LAYERS)
model = Classifier(student_backbone, 'kd').to(DEVICE)
print('Knowledge-distillation teacher:', teacher_repo)
print('Student parameters:', sum(p.numel() for p in model.parameters()))
print('Student trainable:', sum(p.numel() for p in model.parameters() if p.requires_grad))


best_model.pt:   0%|          | 0.00/247M [00:00<?, ?B/s]

Teacher logits:   0%|          | 0/37387 [00:00<?, ?it/s]

Knowledge-distillation teacher: P-261110044/cs781-phase4-a3-full
Student parameters: 45846602
Student trainable: 45846602


## 12. Optimizer and loss

I train using only the training split. Optional class weights come from its labels, not from development or test data.

In [12]:
# Use separate learning rates for the backbone and head; class weights use train labels only.
# ---- Train, keep best dev Macro-F1 checkpoint, then predict ALL official test IDs ----
base_params=[p for n,p in model.named_parameters() if not n.startswith('head.') and p.requires_grad]
groups=[{'params':base_params,'lr':LR_BACKBONE if METHOD in ('full','partial','kd') else LR_ADAPT}] if base_params else []
groups.append({'params':model.head.parameters(),'lr':LR_HEAD})
optimizer=torch.optim.AdamW(groups,weight_decay=.01)

# Compute weights using TRAIN split only — never from dev/test labels.
from sklearn.utils.class_weight import compute_class_weight
train_labels = np.array([y for _, y, _ in train_data], dtype=int)
if USE_CLASS_WEIGHTS and set(train_labels.tolist()) == {0, 1}:
    weights = compute_class_weight('balanced', classes=np.array([0, 1]), y=train_labels)
    CLASS_WEIGHTS = torch.tensor(weights, dtype=torch.float32, device=DEVICE)
else:
    CLASS_WEIGHTS = None
print('Training class weights:', CLASS_WEIGHTS)


Training class weights: tensor([0.8471, 1.2202], device='cuda:0')


## 13. Evaluation

I calculate Macro-F1, macro precision, macro recall and the confusion matrix on labeled data. I use development Macro-F1 to decide which checkpoint to keep.

In [13]:
# Evaluate complete cases without changing the model.
@torch.no_grad()
def evaluate(data):
    model.eval()
    preds=[int(model(parts).argmax()) for parts,_,_ in tqdm(data,desc='Predict',leave=False)]
    if any(y is None for _,y,_ in data): return None,preds
    ys=[int(y) for _,y,_ in data]
    return {'macro_f1':float(f1_score(ys,preds,labels=[0,1],average='macro',zero_division=0)),
             'precision_macro':float(precision_score(ys,preds,labels=[0,1],average='macro',zero_division=0)),
             'recall_macro':float(recall_score(ys,preds,labels=[0,1],average='macro',zero_division=0)),
             'confusion_matrix':confusion_matrix(ys,preds,labels=[0,1]).tolist()}, preds


## 14. Training

I run training and record the loss and development metrics after each epoch. The best checkpoint is saved, and early stopping avoids unnecessary later epochs.

In [14]:
# Save the checkpoint whenever development Macro-F1 improves.
# Reset history when starting a fresh training experiment.
history=[]; best_score=-1; stale=0
best_path=OUT/'best_model.pt'
for ep in range(1,EPOCHS+1):
    model.train(); losses=[]
    order=np.random.RandomState(SEED+ep).permutation(len(train_data))
    optimizer.zero_grad(set_to_none=True)
    for step,j in enumerate(tqdm(order,desc=f'Epoch {ep}'),1):
        j=int(j); parts,y,_=train_data[j]
        logits=model(parts)
        hard=F.cross_entropy(logits.unsqueeze(0),torch.tensor([y],device=DEVICE),weight=CLASS_WEIGHTS)
        if METHOD=='kd':
            teacher_logits=soft_targets[j].to(DEVICE)
            soft=F.kl_div(F.log_softmax(logits/KD_T,dim=-1),
                          F.softmax(teacher_logits/KD_T,dim=-1),reduction='sum')*KD_T**2
            loss=.5*hard+.5*soft
        else: loss=hard
        losses.append(float(loss.detach().cpu()))
        (loss/GRAD_ACCUM).backward()
        if step % GRAD_ACCUM == 0 or step == len(order):
            torch.nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad),1.0)
            optimizer.step();optimizer.zero_grad(set_to_none=True)
    scores,_=evaluate(dev_data)
    row={'epoch':ep,'train_loss':sum(losses)/len(losses),**scores}
    history.append(row)
    (OUT/'history.json').write_text(json.dumps(history,indent=2))
    print('Epoch',ep,'Dev Macro-F1:',scores['macro_f1'])
    if scores['macro_f1'] > best_score:
        best_score=scores['macro_f1'];stale=0
        torch.save({'model':model.state_dict(),'arch':ARCH,'method':METHOD,
                    'chunk_length':CHUNK_LENGTH,'max_chunks':MAX_CHUNKS,
                    'source':P1_REPO,'source_checkpoint':CHECKPOINT_FILE,
                    'dev_macro_f1':best_score},best_path)
    else:
        stale+=1
        if stale>=PATIENCE:break


Epoch 1:   0%|          | 0/37387 [00:00<?, ?it/s]

Predict:   0%|          | 0/1988 [00:00<?, ?it/s]

Epoch 1 Dev Macro-F1: 0.6181305185726342


Epoch 2:   0%|          | 0/37387 [00:00<?, ?it/s]

Predict:   0%|          | 0/1988 [00:00<?, ?it/s]

Epoch 2 Dev Macro-F1: 0.6202172370503793


Epoch 3:   0%|          | 0/37387 [00:00<?, ?it/s]

Predict:   0%|          | 0/1988 [00:00<?, ?it/s]

Epoch 3 Dev Macro-F1: 0.5952430671566991


## 15. Reload the best model

I reload the best checkpoint from disk before the final development evaluation. This avoids accidentally evaluating only the last epoch.

In [15]:
# Restore the best checkpoint rather than using the last epoch automatically.
model.load_state_dict(torch.load(best_path,map_location='cpu',weights_only=True)['model'])
final_dev,_=evaluate(dev_data)


Predict:   0%|          | 0/1988 [00:00<?, ?it/s]

## 16. Test predictions

I predict one binary label for each official test case and check that every expected ID is still present in its original order.

In [16]:
# Produce one test prediction per official case ID.
_,test_predictions=evaluate(test_data)
if any(n is not None for n in (MAX_TRAIN,MAX_DEV,MAX_TEST)):
    raise RuntimeError('Debug caps enabled; disable them and run full data before saving test submission')
records=[{'id':case_id,'pred':int(pred)} for case_id,pred in zip(TEST_IDS,test_predictions)]
assert len(records)==len(test_cases)==len(TEST_IDS)
assert [r['id'] for r in records]==TEST_IDS
assert all(type(r['pred'])==int and r['pred'] in (0,1) for r in records)


Predict:   0%|          | 0/1517 [00:00<?, ?it/s]

## 17. Codabench submission

I save `prediction.json` and put it directly at the root of `submission.zip`, which is the format expected by Codabench.

In [17]:
# The JSON file goes at the root of the submission ZIP.
(OUT/'prediction.json').write_text(json.dumps(records,indent=2),encoding='utf-8')
with zipfile.ZipFile(OUT/'submission.zip','w',zipfile.ZIP_DEFLATED) as z:
    z.write(OUT/'prediction.json',arcname='prediction.json')
with zipfile.ZipFile(OUT/'submission.zip') as z:
    assert z.namelist()==['prediction.json']
    assert json.loads(z.read('prediction.json'))==records


## 18. Save experiment details

I save the run settings, metrics and source-checkpoint details beside the model weights. This makes it easier to compare the separate Kaggle runs later.

In [18]:
# Keep the metrics and settings with the checkpoint.
summary={'arch':ARCH,'method':METHOD,'source':P1_REPO,'source_checkpoint':CHECKPOINT_FILE,
         'train_cases':len(train_cases),'dev_cases':len(dev_cases),'test_cases':len(test_cases),
         'best_dev':final_dev,'parameters':sum(p.numel() for p in model.parameters()),
         'trainable_parameters':sum(p.numel() for p in model.parameters() if p.requires_grad)}
(OUT/'metrics.json').write_text(json.dumps(summary,indent=2))
(OUT/'run_config.json').write_text(json.dumps({'chunk_length':CHUNK_LENGTH,'max_chunks':MAX_CHUNKS,
               'epochs':EPOCHS,'method':METHOD,'teacher_repo':TEACHER_REPO_NAME if METHOD=='kd' else None,
               'seed':SEED,'architecture_config':saved_cfg,'use_class_weights':USE_CLASS_WEIGHTS},indent=2))
if config_name: shutil.copy2(fetch(config_name),OUT/'phase1_architecture_config.json')
(OUT/'README.md').write_text(
    f'# CS781 Phase 4 — {ARCH.upper()} ({METHOD})\n\n'
    f'This experiment uses the Phase 1 checkpoint `{P1_REPO}/{CHECKPOINT_FILE}` '
    f'for case-level binary judgment prediction.\n\n'
    f'I selected the saved model by development Macro-F1 ({best_score:.4f}). '
    'The run folder includes the checkpoint, experiment settings and Codabench predictions.\n'
)
print('VALID submission:',OUT/'submission.zip','Dev metrics:',final_dev)


VALID submission: /kaggle/working/phase4_a3_kd/submission.zip Dev metrics: {'macro_f1': 0.6202172370503793, 'precision_macro': 0.656145323978929, 'recall_macro': 0.6338028169014085, 'confusion_matrix': [[818, 176], [552, 442]]}


## 19. Hugging Face backup

I keep a private Hugging Face repository for this run. Uploading happens only when this cell executes with an authenticated account and `UPLOAD_TO_HF=True`.

In [19]:
# Publish the run folder only after the earlier checks pass.
if UPLOAD_TO_HF:
    api.create_repo(repo_id=OUT_REPO,repo_type='model',private=True,exist_ok=True)
    if not api.repo_info(OUT_REPO,repo_type='model').private:
        raise RuntimeError('Refusing upload: output Hugging Face repo is not private')
    api.upload_folder(repo_id=OUT_REPO,repo_type='model',folder_path=str(OUT),
                      commit_message=f'Phase 4 {ARCH} {METHOD} full experiment and predictions')
    expected={str(f.relative_to(OUT)).replace('\\','/') for f in OUT.rglob('*') if f.is_file()}
    remote=set(api.list_repo_files(OUT_REPO,repo_type='model'))
    missing=expected-remote
    if missing: raise RuntimeError(f'HF upload missing files: {sorted(missing)}')
    print('Hugging Face repo upload VERIFIED: https://huggingface.co/'+OUT_REPO)
else: print('HF upload disabled. Files remain in:',OUT)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/hf_api.py:9662: UserWarning: Warnings while validating metadata in README.md:
- empty or missing yaml metadata in repo card
  warnings.warn(f"Warnings while validating metadata in README.md:\n{message}")


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Hugging Face repo upload VERIFIED: https://huggingface.co/P-261110044/cs781-phase4-a3-kd


## 20. Final file check

Before I finish, I check that the model, metrics, predictions and ZIP were created. A successful local check does not mean Codabench has accepted the submission.

In [20]:
# Confirm the files I expect are present in this run folder.
required = ['best_model.pt', 'metrics.json', 'prediction.json', 'submission.zip', 'run_config.json']
for filename in required:
    file = OUT / filename
    print(('OK     ' if file.is_file() else 'MISSING'), file)
assert all((OUT / name).is_file() for name in required)
print('Output directory:', OUT)
print('HF repo (if uploaded): https://huggingface.co/' + OUT_REPO)
print('Experiment complete: this file trains ONLY the ' + METHOD + ' method.')

OK      /kaggle/working/phase4_a3_kd/best_model.pt
OK      /kaggle/working/phase4_a3_kd/metrics.json
OK      /kaggle/working/phase4_a3_kd/prediction.json
OK      /kaggle/working/phase4_a3_kd/submission.zip
OK      /kaggle/working/phase4_a3_kd/run_config.json
Output directory: /kaggle/working/phase4_a3_kd
HF repo (if uploaded): https://huggingface.co/P-261110044/cs781-phase4-a3-kd
Experiment complete: this file trains ONLY the kd method.
